# Stage 2 → Stage 3 walkthrough: one filing, raw HTML to search results

Every step here calls the project's own functions — nothing is reimplemented — so
what you see is exactly what `fc embed` and `fc build-index` do, one filing at a time.

**Prerequisites**
- `uv sync --group notebook` (ipykernel + pandas), then pick the `.venv` kernel
- Filings downloaded (`fc fetch-filings`), so the EDGAR cache is warm
- Ollama running with `nomic-embed-text` (cells 7–9, 12)
- `fc embed` and `fc build-index` already run, and OpenSearch up (cells 8–12)

Companion reading: `docs/stage2_filing_text.md`, `docs/stage3_embeddings_index.md`.

## 0 · Setup — find the filing in the cache

`download_company` is the same call `fc fetch-filings` makes. With a warm cache it
reads from disk and makes **zero** requests to SEC — asserted below, not assumed.
This gives the real `FilingRef` (accession, report date, path) instead of one typed by hand.

`Settings` is never printed: it holds your EDGAR User-Agent contact, and this repo is public.

In [1]:
import math
import os
from pathlib import Path

# .env, config/ and data/ are relative to the repo root, exactly as when running `fc`.
if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd

from filing_copilot.config import get_settings
from filing_copilot.edgar import EdgarClient
from filing_copilot.filings import download_company
from filing_copilot.structured import Corpus

TICKER = "SYF"          # try COF (item-heading sectioned) or JPM (bundled annual report)
FISCAL_YEAR = 2025

settings = get_settings()
corpus = Corpus.load(settings.corpus_path)
company = corpus.by_ticker(TICKER)

with EdgarClient(settings) as client:
    downloaded = download_company(client, company, years=3)
    assert client.request_count == 0, "cache was cold -- run `fc fetch-filings` first"

document = next(d for d in downloaded.documents if d.ref.fiscal_year == FISCAL_YEAR)
document.ref, document.path

(FilingRef(cik='0001601712', accession='0001601712-26-000006', form='10-K', filing_date=datetime.date(2026, 2, 6), report_date=datetime.date(2025, 12, 31), primary_document='syf-20251231.htm', is_xbrl=True),
 PosixPath('data/raw/filings/1601712/000160171226000006/syf-20251231.htm'))

## 1 · Raw HTML — what SEC actually serves

A 10-K is not a document with some markup around it. It is inline XBRL: every
figure wrapped in a tag, every paragraph in nested spans with inline styles.

In [15]:
raw = document.path.read_bytes()
print(f"{len(raw):,} bytes")
print(f"style= attributes: {raw.count(b'style='):,}   <span: {raw.count(b'<span'):,}   <table: {raw.count(b'<table'):,}")
print()
print(raw[:1500].decode("utf-8", errors="replace"))

3,794,221 bytes
style= attributes: 27,619   <span: 8,534   <table: 124

<?xml version='1.0' encoding='ASCII'?>
<!--XBRL Document Created with the Workiva Platform-->
<!--Copyright 2026 Workiva-->
<!--r:019afef3-c196-7d73-9099-39e2f372c899,g:01b05202-918c-4b13-ada3-baf63dc9a759,d:575160e2e2304bffab18310be1df3aec-->
<html xmlns="http://www.w3.org/1999/xhtml" xmlns:ixt="http://www.xbrl.org/inlineXBRL/transformation/2020-02-12" xmlns:ixt-sec="http://www.sec.gov/inlineXBRL/transformation/2015-08-31" xmlns:ix="http://www.xbrl.org/2013/inlineXBRL" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:dei="http://xbrl.sec.gov/dei/2025" xmlns:us-gaap="http://fasb.org/us-gaap/2025" xmlns:srt="http://fasb.org/srt/2025" xmlns:stpr="http://xbrl.sec.gov/stpr/2025" xmlns:xbrli="http://www.xbrl.org/2003/instance" xmlns:xbrldi="http://xbrl.org/2006/xbrldi" xmlns:iso4217="http://www.xbrl.org/2003/iso4217" xmlns:cyd="http://xbrl.sec.gov/cyd/2025" xmlns:ecd="http://xbrl.sec.gov/ecd/2025" xmlns:syf="

## 2 · Normalize — HTML to one clean string

`normalize()` is pure: bytes in, string out. Data tables become placeholders
(figures come from XBRL via SQL, never from text); layout tables keep their text;
block elements become newlines.

**Every `char_start`/`char_end` in the rest of the system indexes into this exact string.**

In [16]:
from filing_copilot.filings import normalize

text = normalize(raw)
print(f"{len(raw):,} bytes of HTML  ->  {len(text):,} chars of text   ({len(raw) / len(text):.1f}x smaller)")
print(f"data tables replaced by placeholders: {text.count('[TABLE:'):,}")
print()
start = text.find("[TABLE:")
print(text[max(0, start - 600) : start + 200])

3,794,221 bytes of HTML  ->  558,615 chars of text   (6.8x smaller)
data tables replaced by placeholders: 102

nt’s Annual Meeting of Stockholders, to be held June 24, 2026, is incorporated by reference into Part III to the extent described therein.

Synchrony Financial
Table of Contents
OUR ANNUAL REPORT ON FORM 10-K
To improve the readability of this document and better present both our financial results and how we manage our business, we present the content of our Annual Report on Form 10-K in the order listed in the table of contents below. See "Form 10-K Cross-Reference Index" on page 4 for a cross-reference index to the traditional U.S. Securities and Exchange Commission (SEC) Form 10-K format.

[TABLE: 50 rows x 9 cols]

3

Table of Contents

FORM 10-K CROSS REFERENCE INDEX
____________________________________________________________________________________________

Part I

Page(s)

Item 1.



## 3 · Section — where each Item starts and ends

`find_sections` tries item headings first (longest monotonic run of `Item N`
candidates, table-of-contents runs dropped), and falls back to the filer's own
page-reference table. `strategy` records which one produced each section — they are never blended.

In [17]:
from filing_copilot.filings import find_sections

sections = find_sections(text)
pd.DataFrame(
    [
        {"item": s.item, "char_start": s.char_start, "char_end": s.char_end,
         "length": s.length, "strategy": s.strategy, "extra_spans": len(s.extra_spans)}
        for s in sections.values()
    ]
).sort_values("char_start")

,item,char_start,char_end,length,strategy,extra_spans
0,1,62698,127959,65261,crossref_pages,2
6,7,127961,184281,56320,crossref_pages,1
7,7A,184283,192446,8163,crossref_pages,0
1,1A,209139,320070,110931,crossref_pages,1
2,1C,335378,344144,8766,crossref_pages,0
8,8,419132,531695,112563,crossref_pages,0
4,3,528513,531695,3182,crossref_pages,0
9,9A,531698,535789,4091,crossref_pages,0
3,2,535792,537052,1260,crossref_pages,0
5,5,537055,540557,3502,crossref_pages,0


## 4 · One section, read with your own eyes

"Located" only means a span of plausible length was found. Whether it holds the
right content is something only reading it can answer — this is the check `fc show` automates.

In [20]:
risk = sections["1A"]
body = text[risk.char_start : risk.char_end]
print(f"Item 1A: chars {risk.char_start:,}-{risk.char_end:,} ({risk.length:,} long)\n")
print(body[:800])
print("\n[...]\n")
print(body[-500:])
print("\n[...]\n")
print(body)

Item 1A: chars 209,139-320,070 (110,931 long)



Table of Contents

RISKS
Risk Factors Summary
____________________________________________________________________________________________
We are providing the following summary of the risk factors contained in this Annual Report on Form 10-K to enhance the readability and accessibility of our risk factor disclosures. We encourage you to carefully review the full risk factors contained in this Annual Report on Form 10-K in their entirety for additional information regarding the material factors that make an investment in our securities speculative or risky. These risks and uncertainties include, but are not limited to, the following:
Macroeconomic, Strategic and Operational Risks
•Macroeconomic conditions could have a material adverse effect on our business, results of operations and fin

[...]

ows to individuals and entities throughout our Company, including the Board of Directors, various board and management committees and senior man

## 5 · Chunk — ~800-token windows that never cross an item boundary

`item_aware` windows *within* each section; `fixed_window` ignores structure and is
the baseline Stage 4 measures against (ADR-0002). The assert is the property every
citation rests on: a chunk's offsets select exactly its text.

In [21]:
from filing_copilot.filings import FilingText, fixed_window, item_aware

doc = FilingText(
    ref=document.ref, ticker=TICKER, company_name=company.name, text=text, sections=sections
)
chunks = item_aware(doc)

for chunk in chunks:
    assert text[chunk.char_start : chunk.char_end] == chunk.text   # the offset round trip

tokens = pd.Series([c.tokens for c in chunks])
print(f"item_aware: {len(chunks)} chunks   fixed_window: {len(fixed_window(doc))} chunks")
print(f"tokens per chunk: median {tokens.median():.0f}, p95 {tokens.quantile(0.95):.0f}")
print(pd.Series([c.item for c in chunks]).value_counts().sort_index().to_dict())

chunk = next(c for c in chunks if c.item == "1A")
chunk

item_aware: 234 chunks   fixed_window: 255 chunks
tokens per chunk: median 665, p95 793
{'1': 63, '1A': 70, '1C': 5, '2': 1, '3': 1, '5': 2, '7': 32, '7A': 4, '8': 53, '9A': 2, '9B': 1}


Chunk(chunk_id='0001601712-26-000006:1A:209139', cik='0001601712', accession='0001601712-26-000006', form='10-K', period_end=datetime.date(2025, 12, 31), item='1A', section_path='10-K > Item 1A. Risk Factors', char_start=209139, char_end=212263, text="\n\nTable of Contents\n\nRISKS\nRisk Factors Summary\n____________________________________________________________________________________________\nWe are providing the following summary of the risk factors contained in this Annual Report on Form 10-K to enhance the readability and accessibility of our risk factor disclosures. We encourage you to carefully review the full risk factors contained in this Annual Report on Form 10-K in their entirety for additional information regarding the material factors that make an investment in our securities speculative or risky. These risks and uncertainties include, but are not limited to, the following:\nMacroeconomic, Strategic and Operational Risks\n•Macroeconomic conditions could have a material 

## ── Stage 2 ends here. Everything below is Stage 3. ──

## 6 · The embedding input — the exact string the model sees

Three layers, assembled in exactly one place:

```
"search_document: " + contextual_prefix + "\n\n" + chunk.text
```

The task prefix is outermost. nomic needs it, and **Ollama does not add it for you** —
omitting it degrades retrieval with no error at all.

In [22]:
from dataclasses import replace

from filing_copilot.embed import NOMIC_EMBED_TEXT, SEARCH_DOCUMENT, document_input, prepare
from filing_copilot.filings import contextual_prefix

# The same model `fc embed` builds: nomic, at the configured width (768 by default).
model = replace(NOMIC_EMBED_TEXT, dimensions=settings.embedding_dimensions)
print(model)
print("contextual prefix:", contextual_prefix(doc, chunk.item))
print()
body = document_input(doc, chunk)       # prefix + separator + text  (what the pipeline hashes)
sent = prepare(model, SEARCH_DOCUMENT, body)
print(sent[:400])

EmbeddingModel(name='nomic-embed-text', dimensions=768, needs_task_prefix=True, max_input_chars=12000)
contextual prefix: Synchrony Financial (SYF) · 10-K · period ending 2025-12-31 · Item 1A. Risk Factors

search_document: Synchrony Financial (SYF) · 10-K · period ending 2025-12-31 · Item 1A. Risk Factors



Table of Contents

RISKS
Risk Factors Summary
____________________________________________________________________________________________
We are providing the following summary of the risk factors contained in this Annual Report on Form 10-K to enhance the readability and accessibility of our r


## 7 · Encode — 768 numbers, length 1

Ollama returns vectors with a norm around 20. The encoder normalizes every vector to
unit length, because the index scores by inner product — which only equals cosine for unit vectors.

The **digest** is a SHA-256 of `sent` above. It is the vector's key in the cache.

In [ ]:
from filing_copilot.embed import OllamaEncoder, digest

encoder = OllamaEncoder(host=settings.ollama_host, model=model)
live = encoder.encode([body], task=SEARCH_DOCUMENT)[0]
key = digest(model, SEARCH_DOCUMENT, body)

print(f"{len(live)} dimensions, norm {math.sqrt(sum(v * v for v in live)):.6f}")
print("first 8:", [round(v, 4) for v in live[:8]])
print("digest:", key)

768 dimensions, norm 1.000000
first 8: [0.068, 0.0595, -0.2044, 0.0022, 0.0423, -0.0444, 0.0547, -0.0035, 0.0253, -0.0036, -0.0329, 0.0612, 0.0343, -0.0154, 0.0105, -0.0333, 0.0374, 0.0123, -0.0313, 0.0425, -0.0244, -0.0306, -0.0438, -0.015, 0.0328, 0.0832, 0.0053, -0.0204, -0.0262, 0.0226, 0.042, 0.0488, -0.0111, -0.0627, -0.085, -0.0762, 0.0229, 0.0441, 0.0017, -0.0056, 0.018, 0.043, 0.0739, -0.0532, 0.0169, 0.0019, 0.0413, -0.0293, 0.0713, -0.0154, 0.0133, 0.0049, 0.0421, 0.0377, 0.0068, -0.004, 0.0015, 0.0202, -0.0135, -0.0356, 0.0783, 0.0666, -0.0094, 0.0126, 0.0561, -0.0169, -0.081, 0.0485, -0.0078, -0.0203, 0.0507, 0.0064, 0.0362, 0.0231, -0.046, 0.0041, -0.0135, 0.0113, 0.0209, 0.0559, 0.0129, 0.022, 0.0683, 0.0199, 0.0096, -0.0211, -0.0037, -0.0022, 0.0002, 0.0621, 0.0336, 0.018, 0.0224, 0.0499, -0.0472, 0.0582, -0.0298, 0.0088, -0.0338, -0.0317, -0.057, 0.0167, 0.0313, 0.0442, 0.0147, -0.0235, 0.0515, 0.0048, -0.0001, 0.0294, -0.0418, 0.0464, -0.0424, -0.0165, 0.0462, -0.0024

## 8 · The cache holds the same vector

`fc embed` stored this chunk's vector under the same digest. Cosine ≈ 1.0 shows the
cache is exactly what the model says — and why a changed input can never be served a stale vector:
a different input is a different digest.

In [26]:
from filing_copilot.embed import EmbeddingCache

cache = EmbeddingCache(settings.embeddings_dir)
cached = cache.load(model, [key])[key]
print("cosine(live, cached) =", round(sum(a * b for a, b in zip(live, cached)), 6))
cache.stats(model)

cosine(live, cached) = 1.0


CacheStats(model='nomic-embed-text@768', vectors=14043, dimensions=768, parts=55, total_bytes=68397562)

## 9 · Semantic search by hand — what k-NN does before OpenSearch

Embed a *question* (note: `search_query`, not `search_document`), then take the dot
product with every one of this company's cached vectors and sort. That is all
nearest-neighbour search is. OpenSearch's HNSW index does the same thing approximately,
without comparing against every vector.

In [27]:
from filing_copilot.embed import SEARCH_QUERY
from filing_copilot.index import manifest_path, read_manifest

QUESTION = "credit card net charge-off risk"
query_vector = encoder.encode([QUESTION], task=SEARCH_QUERY)[0]

rows = read_manifest(manifest_path(settings.manifests_dir, "item_aware", model.cache_key))
company_rows = [r for r in rows if r.ticker == TICKER]
vectors = cache.load(model, {r.digest for r in company_rows})

scored = sorted(
    ((sum(a * b for a, b in zip(query_vector, vectors[r.digest])), r) for r in company_rows),
    key=lambda pair: pair[0],
    reverse=True,
)
by_hand = pd.DataFrame(
    [{"cosine": round(s, 4), "fy": r.fiscal_year, "item": r.item, "chunk_id": r.chunk_id,
      "text": r.text[:90]} for s, r in scored[:5]]
)
by_hand

,cosine,fy,item,chunk_id,text
0,0.6715,2023,1A,0001601712-24-000047:1A:241774,neral purpose co-branded credit cards and priv...
1,0.6564,2024,1A,0001601712-25-000044:1A:247839,e co-branded credit cards and private label cr...
2,0.6538,2025,8,0001601712-26-000006:8:456027,he same loan receivable may meet more than one...
3,0.6476,2025,1A,0001601712-26-000006:1A:294408,"nt card networks may vary, the fee paid to the..."
4,0.6445,2023,1,0001601712-24-000047:1:97373,tical component of our management and growth s...


## 10 · The manifest — Stage 3's durable output

One row per chunk: metadata, the chunk text (without the contextual prefix — that
is for the encoder only), and the digest linking it to its vector. Together with the cache,
this is everything needed to rebuild the index. The index itself is disposable.

In [28]:
manifest = pd.DataFrame(rows)
print(f"{len(manifest):,} rows, {manifest['digest'].nunique():,} distinct vectors")
manifest.pivot_table(index="ticker", columns="item", values="chunk_id", aggfunc="count", fill_value=0)

14,043 rows, 14,043 distinct vectors


item,1,10,11,12,13,14,15,16,1A,1C,2,3,4,5,7,7A,8,9A,9B
ticker,,,,,,,,,,,,,,,,,,,
AAPL,21,9,0,0,0,0,0,3,98,2,3,6,0,3,20,3,57,6,3
ALLY,111,16,3,3,3,3,3,3,175,0,3,0,0,3,410,3,375,3,3
AXP,123,0,0,0,0,3,4,12,152,13,3,0,0,6,200,0,217,3,6
BAC,52,9,3,3,3,3,6,3,175,0,3,0,0,3,333,3,393,3,3
BFH,124,0,0,0,0,0,0,189,212,9,3,3,0,5,107,0,0,24,0
CFG,97,3,3,3,3,3,15,6,85,8,3,0,0,3,141,3,233,3,3
COF,110,3,3,3,3,3,3,21,195,15,3,3,0,3,258,325,0,4,3
FITB,68,3,3,3,3,3,3,7,124,12,3,3,9,6,263,3,386,10,3
JEF,55,3,3,3,3,3,3,9,71,6,3,3,0,3,164,3,329,3,3


## 11 · One index document — manifest row + vector

This is exactly the JSON `build_index` sends to OpenSearch for one chunk
(vector shortened for display). `chunk_id` becomes the document `_id`, so rebuilding overwrites instead of duplicating.

In [29]:
from filing_copilot.index import to_document

row = next(r for r in rows if r.digest == key)
document_json = to_document(row, vectors.get(key, cached))
{**document_json, "embedding": document_json["embedding"][:4] + ["..."], "text": row.text[:120] + "..."}

{'chunk_id': '0001601712-26-000006:1A:209139',
 'cik': '0001601712',
 'ticker': 'SYF',
 'accession': '0001601712-26-000006',
 'form': '10-K',
 'period_end': '2025-12-31',
 'fiscal_year': 2025,
 'item': '1A',
 'section_path': '10-K > Item 1A. Risk Factors',
 'char_start': 209139,
 'char_end': 212263,
 'text': '\n\nTable of Contents\n\nRISKS\nRisk Factors Summary\n________________________________________________________________________...',
 'digest': '2db57dcb6eac8b1a5c686ec8dc118b47031d3b10ad78a054a389d19581a91b99',
 'model': 'nomic-embed-text@768',
 'embedding': [0.06795084476470947,
  0.059471823275089264,
  -0.204423725605011,
  0.0022398510482162237,
  '...']}

## 12 · Query the index — lexical, semantic, and filtered

- **BM25** matches words ("CECL", "Item 9A") — the half that doesn't paraphrase.
- **k-NN** matches meaning. faiss reports inner product as `1 + dot`, so subtract 1 to compare with cell 9.
- **k-NN + filter** scopes to one company. Its top 5 should match cell 9's hand-computed list.

Stage 4 fuses BM25 and k-NN by *rank* (RRF, ADR-0010), because their scores are on unrelated scales.

In [30]:
import httpx
from filing_copilot.index import index_name

url = f"{settings.opensearch_url}/{index_name(settings.opensearch_index_prefix, 'item_aware')}/_search"
fields = ["ticker", "fiscal_year", "item", "chunk_id", "text"]

def search(query: dict, size: int = 5) -> pd.DataFrame:
    hits = httpx.post(url, json={"size": size, "_source": fields, "query": query}).json()["hits"]["hits"]
    return pd.DataFrame(
        [{"score": round(h["_score"], 4), "fy": h["_source"]["fiscal_year"], "ticker": h["_source"]["ticker"],
          "item": h["_source"]["item"], "chunk_id": h["_id"], "text": h["_source"]["text"][:90]} for h in hits]
    )

print("BM25 ----------------------------------------------------------")
display(search({"match": {"text": QUESTION}}))
print("k-NN, whole corpus ---------------------------------------------")
display(search({"knn": {"embedding": {"vector": query_vector, "k": 5}}}))
print(f"k-NN, ticker = {TICKER} ------------------------------------------")
filtered = search({"knn": {"embedding": {"vector": query_vector, "k": 5, "filter": {"term": {"ticker": TICKER}}}}})
display(filtered)

print("matches the by-hand ranking from cell 9:", list(filtered["chunk_id"]) == list(by_hand["chunk_id"]))

BM25 ----------------------------------------------------------


,score,fy,ticker,item,chunk_id,text
0,15.0572,2025,SOFI,7,0001818874-26-000013:7:666256,Excludes the impact of delinquent personal loa...
1,14.5478,2024,SOFI,7,0001818874-25-000016:7:623024,ision for credit losses.\n(4)Excludes the impa...
2,14.5137,2025,SOFI,7,0001818874-26-000013:7:661187,cross SoFi Money and Credit Card and brokerage...
3,13.9371,2025,BFH,16,0001101215-26-000016:16:451756,and Net principal loss rates are also impacte...
4,13.6979,2024,COF,7,0000927628-25-000092:7:502541,luded in our consolidated statements of income...


k-NN, whole corpus ---------------------------------------------


,score,fy,ticker,item,chunk_id,text
0,1.7482,2025,CFG,7,0000759944-26-000028:7:331963,mmercial.\nThe following table presents the ne...
1,1.6900,2024,COF,7,0000927628-25-000092:7:502541,luded in our consolidated statements of income...
2,1.6886,2025,COF,1A,0000927628-26-000024:1A:378655,surcharges or other point of sale pricing stra...
3,1.6878,2025,COF,7,0000927628-26-000024:7:563762,coveries)\n\n[TABLE: 20 rows x 47 cols]\n_____...
4,1.6872,2024,CFG,7,0000759944-25-000013:7:335878,it Risk Function is responsible for reviewing ...


k-NN, ticker = SYF ------------------------------------------


,score,fy,ticker,item,chunk_id,text
0,1.6715,2023,SYF,1A,0001601712-24-000047:1A:241774,neral purpose co-branded credit cards and priv...
1,1.6564,2024,SYF,1A,0001601712-25-000044:1A:247839,e co-branded credit cards and private label cr...
2,1.6538,2025,SYF,8,0001601712-26-000006:8:456027,he same loan receivable may meet more than one...
3,1.6476,2025,SYF,1A,0001601712-26-000006:1A:294408,"nt card networks may vary, the fee paid to the..."
4,1.6445,2023,SYF,1,0001601712-24-000047:1:97373,tical component of our management and growth s...


matches the by-hand ranking from cell 9: True


## What Stage 3 outputs

| Artifact | Durable? | Rebuilt by |
|---|---|---|
| Embedding cache — `(digest, vector)` Parquet parts | yes, append-only | `fc embed` (only missing digests) |
| Manifest — one row per chunk, with its digest | yes, replaced each run | `fc embed` |
| OpenSearch index | **no** — derived | `fc build-index`, ~14s, zero embeddings |

The index is the join of the other two on `digest`. Lose it and nothing is lost.
Stage 4 queries it both ways and fuses the rankings.

In [31]:
encoder.close()